## Instructions ⚠️

### For best use of this code during LASC, follow the steps below. Happy simulating!

### 1) Modify simulation parameters ONLY through the User Edit Panel, which can be easily found using the VsCode "Outline" in the left sidebar.

### 2) For easier visualization, all Monte Carlo images and plots will be saved to a PDF. It can be accessed from the left sidebar after generating it.

# 

## Lib import

In [ ]:
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import threading

from rocketpy import Environment, Flight, MonteCarlo, Rocket, SolidMotor

from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticFlight,
    StochasticNoseCone,
    StochasticRailButtons,
    StochasticRocket,
    StochasticSolidMotor,
    StochasticTail,
    StochasticTrapezoidalFins,
    StochasticParachute,
)

## Launch Site

### Code section where the launch site and environment variables are defined

In [ ]:
env = Environment(
    latitude = -21.9419,                # LASC latitude
    longitude= -48.9531,                # LASC longitude
    timezone = 'America/Sao_Paulo',     # Timezone
    datum="WGS84"                       # Coordinate source (global GPS standard)
)

DATETIME = datetime.datetime.fromisoformat("2026-06-23 12:00:00")       # Launch datetime (used to fetch weather conditions)
env.set_date(DATETIME)


result = {"success": False, "error": None}

def download_gefs():                                                    # Function to fetch Ensemble - GEFS atmospheric data
    try:
        env.set_atmospheric_model(type="Ensemble", file="GEFS")         # Atmospheric model (Ensemble is the best choice for Monte Carlo)
        result["success"] = True
    except Exception as e:
        result["error"] = e

thread = threading.Thread(target=download_gefs)                         # Since GEFS files are large, a backup atmospheric model
thread.start()                                                          # is required in case the Ensemble download fails
thread.join(timeout=90)  # waits up to 90 seconds 

if thread.is_alive():
    print("GEFS did not respond — using ECMWF as fallback")
    env.set_atmospheric_model(type="Windy", file="ECMWF")
elif result["success"]:
    print("GEFS loaded successfully")
else:
    print(f"GEFS returned an error: {result['error']} — using ECMWF")
    env.set_atmospheric_model(type="Windy", file="ECMWF")


env.set_topographic_profile(type="NASADEM_HGT", file="NASADEM_NC_s22w049.nc", dictionary="netCDF4", crs=None)       # Topographic model

elevation = env.get_elevation_from_topographic_profile(env.latitude, env.longitude)

env.set_elevation(elevation)


env.all_info()      # Display all environment information

### Ground elevation retrieved from the NASA API

## Stockhastic Launch Site

### Definition of parameter deviations for the launch environment in the Monte Carlo simulation

In [ ]:
if env.atmospheric_model_type == "Ensemble":                    # Checks the atmospheric model type to set ensemble members
    ensemble_member=list(range(env.num_ensemble_members))       # List of probable forecasts for the launch site
else:
    ensemble_member=None


stochastic_env = StochasticEnvironment(
    environment=env,
    
    ensemble_member=ensemble_member,

    wind_velocity_x_factor=(1.0, 0.1),      # Multiplicative factor for wind velocity in the x direction, standard deviation
    wind_velocity_y_factor=(1.0, 0.1),      # Multiplicative factor for wind velocity in the y direction, standard deviation
)

stochastic_env.visualize_attributes()

## Atlas Motor

### Definition of the Atlas motor

In [ ]:
Proton = SolidMotor(
    thrust_source="PROTON_teste_estaticonov105CM.eng",       # Motor .eng file
    dry_mass = 8.88,               
    dry_inertia=(1.065, 1.065, 0.02377),
    nozzle_radius= 32.27 / 1000,
    grain_number= 6,
    grain_density= 1750.00622798,
    grain_outer_radius= 45 / 1000,
    grain_initial_inner_radius= 19.05 / 1000,
    grain_initial_height= 140 / 1000,
    grain_separation= 12 / 1000,
    grains_center_of_mass_position= 545.10 / 1000,      # From Fusion (Motor length - CM with grains only -- Origin is inverted in CAD)
    center_of_dry_mass_position= 521.05 / 1000,         # From Fusion (Motor length - CM) -- Origin is inverted in CAD
    nozzle_position= 0 /1000,                           # Position of the nozzle exit
    throat_radius= 13.5 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

Proton.info()

## Stockhastic Atlas Motor

### Definition of parameter deviations for the Atlas motor in the Monte Carlo simulation

In [ ]:
stochastic_Motor = StochasticSolidMotor(
    solid_motor=Proton,
    burn_start_time=(0, 0.01, "binomial"), # Mean, Standard Deviation, distribution type
    grains_center_of_mass_position= 0,
    grain_density= 0,
    grain_separation= 0 / 1000,
    grain_initial_height= 0 / 1000,
    grain_initial_inner_radius= 0 / 1000,
    grain_outer_radius= 0 / 1000,
    total_impulse= 200,
    throat_radius= 0 / 1000,  
    nozzle_radius= 0 / 1000, 
    nozzle_position= 0,
)
stochastic_Motor.visualize_attributes()

#### Deviations confirmed by the propulsion team

## Rocket structure

### Definition of the full structural configuration of the Atlas rocket

In [ ]:
Atlas = Rocket(
    radius= 76/1000,
    mass= 14332/1000,
    inertia=(5.549, 5.549, 0.05654),                 # From Assembly
    power_off_drag="Atlas_CD_Power-Off.csv",         # From RASAero
    power_on_drag="Atlas_CD_Power-On.csv",           # From RASAero
    center_of_mass_without_motor= 1294/1000,
    coordinate_system_orientation="nose_to_tail",
)

Nosecone = Atlas.add_nose(
    length= 850/1000, kind="Von Karman", position=0
)

Fins = Atlas.add_trapezoidal_fins(
    n=4,
    root_chord = 290/1000,
    tip_chord = 52/1000,
    span = 142.1/1000,
    position = 2417/1000,
    cant_angle = 0,
    sweep_length = 228/1000,
)


Atlas.add_motor(Proton, position= 2702/1000)


boattail = Atlas.add_tail(
    top_radius=76/1000, bottom_radius=64.5/1000, length=300/1000, position=2407/1000, name="Conical Boattail",
)

rail_buttons = Atlas.set_rail_buttons(
    upper_button_position= 1630/1000,
    lower_button_position= 2382/1000,
    angular_position=45,
)

main_parachute = Atlas.add_parachute( 
    name="Main Parachute Atlas",
    cd_s=5.193,     # cd: 0.9 x s: 5.77 m^2 (REC team will update these values)
    trigger= "apogee",   # ejection altitude (m)
    sampling_rate=50,   # barometer update rate in Hz (confirmed by avionics)
    lag=0.118,      # parachute opening delay (calculated by REC team)
)


## Stockhastic rocket structure

### Definition of parameter deviations for the Atlas structure in the Monte Carlo simulation

In [ ]:
stochastic_Atlas = StochasticRocket(
    rocket=Atlas,
    radius= 0.005,
    mass=(14332/1000, 200/1000, "normal"), # Mean, Standard Deviation, distribution type
    inertia_11=(5.549, 0.001),
    inertia_22=0.001,
    inertia_33=0.0001,
    center_of_mass_without_motor=0.005,
    power_off_drag_factor=(1, 0.05),  # Multiplier for rocket's drag curve. Usually has a mean value of 1
    # and an uncertainty of 5% to 10%

    power_on_drag_factor=(1, 0.05),   # Multiplier for rocket's drag curve. Usually has a mean value of 1
    # and an uncertainty of 5% to 10%
)

stochastic_Nosecone = StochasticNoseCone(
    nosecone=Nosecone,
    length=0.001,
)

stochastic_Fins = StochasticTrapezoidalFins(
    trapezoidal_fins=Fins,
    root_chord=0.0005,
    tip_chord=0.0005,
    span=0.0005,
)

stochastic_boattail = StochasticTail(
    tail=boattail,
    top_radius=0.0005,
    bottom_radius=0.0005,
    length=0.001,
)

stochastic_rail_buttons = StochasticRailButtons(
    rail_buttons=rail_buttons, buttons_distance=0.0005
)

stochastic_main_parachute = StochasticParachute(
    parachute=main_parachute,
    cd_s=0.2,   # Recommended by REC team
    lag=0.032,  # Calculated by REC team
)


stochastic_Atlas.add_motor(stochastic_Motor, position=0.001)
stochastic_Atlas.add_nose(stochastic_Nosecone, position=(0, 0.001))
stochastic_Atlas.add_trapezoidal_fins(stochastic_Fins, position=(0.001, "normal"))
stochastic_Atlas.add_tail(stochastic_boattail)
stochastic_Atlas.set_rail_buttons(stochastic_rail_buttons, lower_button_position=(0.0005, "normal"))
stochastic_Atlas.add_parachute(stochastic_main_parachute)

stochastic_Atlas.visualize_attributes()
#stochastic_Fins.visualize_attributes()
#stochastic_coifa.visualize_attributes()

#### Deviations confirmed by the structures team

## Flight Conditions

### Definition of flight conditions at the launch pad

In [ ]:
flightStage = Flight(
    rocket=Atlas,
    environment=env,
    rail_length=6,  # meters
    inclination=80, # degrees
    heading=90,     # degrees
)

## Stockhastic flight conditions

### Definition of parameter deviations for the launch pad flight conditions in the Monte Carlo simulation

In [ ]:
stochastic_flight = StochasticFlight(
    flight=flightStage,
    rail_length=(6, 0.01),
    inclination=(80, 1),  # angle, standard deviation
    heading=(90, 2),     # angle, standard deviation
)

stochastic_flight.visualize_attributes()

<h1 style="color: orange; font-size: xxx-large">USER EDIT PANEL ⚙️</h1>

In [ ]:
exec(open('code_edition_interface.py', encoding='utf-8').read())

## Visual configuration

### Visual check of rocket components as defined in the code

In [ ]:
Proton.draw()
Proton.draw(filename="1Proton.png")

Fins.draw()
Fins.draw(filename="1Fins.png")

Atlas.draw()
Atlas.draw(filename="1Atlas.png")


## Monte Carlo Simulation

In [ ]:
Simulation = MonteCarlo(
    filename="Resultados_Monte-Carlo",
    environment=stochastic_env,
    rocket=stochastic_Atlas,
    flight=stochastic_flight,
)

Simulation.simulate(
    number_of_simulations=1000,      # Run 1000
    append=False,
    include_function_data=False,
    parallel=True,
    n_workers=None,
)

## Results

### Flight variable values (Deterministic)

In [ ]:
Simulation.prints.all()

flightStage.prints.maximum_values()

### Saving results to a dictionary

In [ ]:
simulation_general_results = []

simulation_results = {
    "out_of_rail_time": [],
    "out_of_rail_velocity": [],
    "apogee_time": [],
    "apogee": [],
    "apogee_x": [],
    "apogee_y": [],
    "t_final": [],
    "x_impact": [],
    "y_impact": [],
    "impact_velocity": [],
    "initial_stability_margin": [],
    "out_of_rail_stability_margin": [],
    "max_mach_number": [],
    "frontal_surface_wind": [],
    "lateral_surface_wind": [],
    "index": [],
}

simulation_output_file = open(str("Resultados_Monte-Carlo") + ".outputs.txt", "r+")

# Read each line of the file and convert to dict
for line in simulation_output_file:
    # Skip comments lines
    if line[0] != "{":
        continue
    # Eval results and store them
    flight_result = eval(line)
    simulation_general_results.append(flight_result)
    for parameter_key, parameter_value in flight_result.items():
        simulation_results[parameter_key].append(parameter_value)

# Close data file
simulation_output_file.close()

# Print number of flights simulated
N = len(simulation_general_results)

### Apogee (Monte Carlo)

In [ ]:
print(
    f"Apogee Altitude - Mean Value: {np.mean(simulation_results['apogee']):0.3f} m"
)
print(
    f"Apogee Altitude - Standard Deviation: {np.std(simulation_results['apogee']):0.3f} m"
)

plt.figure()
plt.hist(simulation_results["apogee"], bins=int(N**0.5))
plt.title("Apogee Altitude")
plt.xlabel("Altitude (m)")
plt.ylabel("Number of Occurences")
plt.show()

# ---------------------------------------------------

print(
    f"Apogee Time - Mean Value: {np.mean(simulation_results['apogee_time']):0.3f} s"
)
print(
    f"Apogee Time - Standard Deviation: {np.std(simulation_results['apogee_time']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["apogee_time"], bins=int(N**0.5))
plt.title("Apogee Time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

### Stability margin (Deterministic + Monte Carlo)

In [ ]:
Atlas.plots.static_margin()

# ---------------------------------------

print(
    f"Stability Margin - Mean Value: {np.mean(simulation_results['initial_stability_margin']):0.3f} cal"
)
print(
    f"Stability Margin - Standard Deviation: {np.std(simulation_results['initial_stability_margin']):0.3f} cal"
)

plt.figure()
counts, edges, patches = plt.hist(simulation_results["initial_stability_margin"], bins=int(N**0.5))
plt.title("Stability Margin")
plt.xlabel("Cal")
plt.ylabel("Number of Occurences")
plt.xticks(edges, labels=[f"{e:.2f}" for e in edges])
plt.tight_layout()
plt.show()


### Wind velocities / Ground impact velocity (Monte Carlo)

In [ ]:
print(
    f"Frontal Surface Wind - Mean Value: {np.mean(simulation_results['frontal_surface_wind']):0.3f} m/s"
)
print(
    f"Frontal Surface Wind - Standard Deviation: {np.std(simulation_results['frontal_surface_wind']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["frontal_surface_wind"], bins=int(N**0.5))
plt.title("Frontal Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Lateral Surface Wind - Mean Value: {np.mean(simulation_results['lateral_surface_wind']):0.3f} m/s"
)
print(
    f"Lateral Surface Wind - Standard Deviation: {np.std(simulation_results['lateral_surface_wind']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["lateral_surface_wind"], bins=int(N**0.5))
plt.title("Lateral Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Impact Velocity - Mean Value: {np.mean(simulation_results['impact_velocity']):0.3f} m/s"
)
print(
    f"Impact Velocity - Standard Deviation: {np.std(simulation_results['impact_velocity']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["impact_velocity"], bins=int(N**0.5))
plt.title("Impact Velocity")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()


### Velocity (Deterministic), Max Mach and Flight time (Monte Carlo)

In [ ]:
flightStage.speed.plot(0, flightStage.apogee_time)

# ----------------------------------------------------------------

print(
    f"Flight time - Mean Value: {np.mean(simulation_results['t_final']):0.3f} s"
)
print(
    f"Flight time - Standard Deviation: {np.std(simulation_results['t_final']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["t_final"], bins=int(N**0.5))
plt.title("Flight time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Max Mach Number - Mean Value: {np.mean(simulation_results['max_mach_number']):0.3f} "
)
print(
    f"Max Mach Number - Standard Deviation: {np.std(simulation_results['max_mach_number']):0.3f} "
)

plt.figure()
plt.hist(simulation_results["max_mach_number"], bins=int(N**0.5))
plt.title("Max Mach Number")
plt.xlabel("Mach")
plt.ylabel("Number of Occurences")
plt.show()

### Probable landing radius (Monte Carlo)

In [ ]:
Simulation.plots.ellipses(xlim=(-9000, 9000), ylim=(-9000, 9000)) # Visualization of the landing radius
Simulation.plots.ellipses(save=True)

fig = plt.gcf()
fig.set_size_inches(24, 24)  # increases figure size

ax = plt.gca()
ax.set_xlim(-9000, 9000)
ax.set_ylim(-9000, 9000) 
ax.xaxis.set_major_locator(plt.MultipleLocator(500))    # 500m spacing on axes
ax.yaxis.set_major_locator(plt.MultipleLocator(500))
plt.savefig("Resultados_Monte-Carlo.png")
plt.show()

# image="Launch Site 2km_x_2km.png" can be passed as argument to "ellipses" to add a background image

### 3D flight trajectory

In [ ]:
flightStage.plots.trajectory_3d()

flightStage.plots.trajectory_3d(filename="1Trajectory_Stage.png")


### Saving all plots to a PDF

In [ ]:
plt.ioff()

with PdfPages('Plots_de_Resultados.pdf') as pdf:

    fig = plt.figure(figsize=(8, 8)) 
    img_P = plt.imread('1Proton.png')
    plt.imshow(img_P)
    plt.axis('off')
    plt.title('Proton')
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 8)) 
    img_F = plt.imread('1Fins.png')
    plt.imshow(img_F)
    plt.axis('off')
    plt.title('Fins')
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 8)) 
    img_A = plt.imread('1Atlas.png')
    plt.imshow(img_A)
    plt.axis('off')
    plt.title('Atlas')
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["apogee"], bins=int(N**0.5))
    plt.title("Apogee Altitude")
    plt.xlabel("Altitude (m)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["apogee_time"], bins=int(N**0.5))
    plt.title("Apogee Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    counts, edges, patches = plt.hist(simulation_results["initial_stability_margin"], bins=int(N**0.5))
    plt.title("Stability Margin")
    plt.xlabel("Cal")
    plt.ylabel("Number of Occurences")
    plt.xticks(edges, labels=[f"{e:.2f}" for e in edges])
    plt.tight_layout()
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["frontal_surface_wind"], bins=int(N**0.5))
    plt.title("Frontal Surface Wind")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["lateral_surface_wind"], bins=int(N**0.5))
    plt.title("Lateral Surface Wind")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["impact_velocity"], bins=int(N**0.5))
    plt.title("Impact Velocity")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["t_final"], bins=int(N**0.5))
    plt.title("Flight time")
    plt.xlabel("Time (s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 6)) 
    plt.hist(simulation_results["max_mach_number"], bins=int(N**0.5))
    plt.title("Max Mach Number")
    plt.xlabel("Mach")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(24, 24))                      # Adjusts image size for the PDF
    img_E = plt.imread('Resultados_Monte-Carlo.png')
    plt.imshow(img_E)
    plt.axis('off')
    plt.title('Landing Radius')
    pdf.savefig()
    plt.close()

    fig = plt.figure(figsize=(8, 8)) 
    img_P = plt.imread('1Trajectory_Stage.png')
    plt.imshow(img_P)
    plt.axis('off')
    plt.title('3D Trajectory')
    pdf.savefig()
    plt.close()


In [ ]:
flightStage.export_kml(
    file_name = "Atlas.kml",
    extrude = True,
    altitude_mode = "relativetoground",
)